# 27 — Configuration, Display Customization, & Copy-on-Write Settings
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to configuring Pandas runtime options, scoping display settings via context managers, and mastering the modern Copy-on-Write engine.*

---

## 📌 Executive Summary & Interview Expectations
Managing environment state and display formatting is crucial in both production data pipelines and executive reporting. In technical interviews, examiners evaluate:
1. **Global Mutation vs Scoped Context Managers**: Why `pd.set_option()` is dangerous in shared codebases and how `pd.option_context()` guarantees thread-safe, reversible styling.
2. **Display Diagnostics**: Configuring `max_rows`, `max_columns`, `max_colwidth`, and `precision` to inspect wide and deep datasets without truncation.
3. **Floating-Point Formatting vs Value Mutation**: Understanding that `display.precision` and `display.float_format` alter visual presentation **without corrupting the underlying floating-point precision**.
4. **The Copy-on-Write (CoW) Architectural Revolution**: How Pandas 2.0+ / 3.0 `mode.copy_on_write` eliminates `SettingWithCopyWarning`, prevents defensive copies, and optimizes memory overhead.

## 1. Environment Setup & Data Ingestion

In [1]:
import numpy as np
import pandas as pd

# Load World Happiness Report
happiness = pd.read_csv("happiness.csv")
print(f"Happiness Dataset Loaded: {happiness.shape[0]} countries, {happiness.shape[1]} columns")
happiness.head(3)

Happiness Dataset Loaded: 156 countries, 6 columns


,Country,Score,GDP per capita,Social support,Life expectancy,Generosity
0,Finland,7.769,1.340,1.587,0.986,0.153
1,Denmark,7.600,1.383,1.573,0.996,0.252
2,Norway,7.554,1.488,1.582,1.028,0.271


## 2. Option Discovery & Inspection (`get_option` & `describe_option`)

### API Equivalence:
- Functional: `pd.get_option("display.max_rows")`
- Attribute-based: `pd.options.display.max_rows`
- Documentation: `pd.describe_option("max_col")` prints the option description and default value directly!

In [2]:
# Inspect current default display limits
print("Default max_rows:", pd.get_option("display.max_rows"))
print("Default max_columns:", pd.get_option("display.max_columns"))
print("Default precision:", pd.get_option("display.precision"))

# Search and print option documentation dynamically
print("\n--- Option Documentation for 'display.max_colwidth' ---")
pd.describe_option("display.max_colwidth")

Default max_rows: 60
Default max_columns: 20
Default precision: 6

--- Option Documentation for 'display.max_colwidth' ---
display.max_colwidth : int or None
    The maximum width in characters of a column in the repr of
    a pandas data structure. When the column overflows, a "..."
    placeholder is embedded in the output. A 'None' value means unlimited.
    [default: 50] [currently: 50]


## 3. Formatting Numbers: `precision` & `chop_threshold`

### 💡 Interview Tip: Visual Formatting vs Data Corruption
- `pd.set_option('display.precision', 2)` affects **only the string representation** in notebook outputs.
- It does **NOT round or truncate the underlying 64-bit float numbers** in memory.
- `display.chop_threshold`: Automatically displays values smaller than threshold as `0` (suppresses tiny floating point artifacts like `1e-17`).

In [3]:
raw_score = happiness.loc[0, "Score"]
print(f"Raw Score float in memory: {raw_score} (Full IEEE-754 precision)")

# Scope precision changes using option_context
with pd.option_context("display.precision", 2):
    print("Formatted view (precision=2):")
    display(happiness[["Country", "Score", "GDP per capita"]].head(3))

print(f"Value in memory remains unaltered: {happiness.loc[0, 'Score']}")

Raw Score float in memory: 7.769 (Full IEEE-754 precision)
Formatted view (precision=2):


,Country,Score,GDP per capita
0,Finland,7.77,1.34
1,Denmark,7.60,1.38
2,Norway,7.55,1.49


Value in memory remains unaltered: 7.769


## 4. Production Standards: The `pd.option_context()` Pattern

### ⚠️ Top Interview Anti-Pattern: Mutating Global State in Shared Libraries
```python
# Junior Approach (Mutates global interpreter state permanently for all downstream users):
pd.set_option('display.max_rows', 500)
df.head(500)

# Senior Production Standard (Scoped context manager, automatically reverts upon exit):
with pd.option_context('display.max_rows', 500, 'display.max_columns', None):
    display(df)
```

In [4]:
# Demonstrate safe multi-option scoping
with pd.option_context(
    "display.max_rows", 6,
    "display.max_columns", 4,
    "display.precision", 3
):
    print("Inside Context (Restricted rows/columns):")
    display(happiness)

print("\nOutside Context (Reverted automatically):")
print("max_rows is safely back to:", pd.get_option("display.max_rows"))

Inside Context (Restricted rows/columns):


,Country,Score,...,Life expectancy,Generosity
0,Finland,7.769,...,0.986,0.153
1,Denmark,7.600,...,0.996,0.252
2,Norway,7.554,...,1.028,0.271
...,...,...,...,...,...
153,Afghanistan,3.203,...,0.361,0.158
154,Central African Republic,3.083,...,0.105,0.235
155,South Sudan,2.853,...,0.295,0.202



Outside Context (Reverted automatically):
max_rows is safely back to: 60


## 5. The Copy-on-Write (CoW) Engine in Pandas 2.x & 3.0

### 🚨 Major Technical Interview Topic: How Copy-on-Write Works
In legacy Pandas, taking a slice (`subset = df[cols]`) could either return a view or a copy, leading to erratic `SettingWithCopyWarning` and unpredictable chained assignment bugs.

Under **Copy-on-Write (`pd.set_option('mode.copy_on_write', True)`)**:
1. Any slice or indexing operation behaves like a **read-only view without copying memory**.
2. Memory is **only copied when you attempt to modify** the data!
3. Eliminates defensive copying, vastly reducing memory usage in large pipelines.

In [5]:
# Inspect Copy-on-Write status
cow_status = pd.get_option("mode.copy_on_write")
print(f"Current Copy-on-Write mode: {cow_status}")

# Safe demonstration of CoW behavior
with pd.option_context("mode.copy_on_write", True):
    base_df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]})
    subset_df = base_df[["A"]]  # Shares memory under CoW
    
    # Modifying subset triggers copy ON WRITE, protecting base_df!
    subset_df.iloc[0, 0] = 999
    
    print("Modified subset_df:")
    print(subset_df)
    print("Original base_df (Remains completely unchanged!):")
    print(base_df)

Current Copy-on-Write mode: True
Modified subset_df:
     A
0  999
1    2
2    3
Original base_df (Remains completely unchanged!):
   A  B
0  1  4
1  2  5
2  3  6


/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_27520/4123662367.py:2: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  cow_status = pd.get_option("mode.copy_on_write")


---
## 🎯 6. Technical Interview Corner: Tricky Questions & Drills

### Q1: Why is mutating `pd.set_option()` considered a code smell in production library packages?
**Answer**:
When writing a utility package or library imported by other services, calling `pd.set_option()` alters the global singleton configuration of the entire Python process. If a library function sets `pd.set_option('display.max_columns', 1000)`, it can cause memory bloat, log flooding, or break display assumptions in caller notebooks and automated reports. Always encapsulate configuration changes within `pd.option_context()`!

---

### Q2: Advanced Interview Coding Challenge: Custom Currency & Percentage Formatting
**Challenge**:
Format the `happiness` report for an executive dashboard:
- Format `'Score'` to 2 decimal places with a `'★'` suffix.
- Format `'GDP per capita'` as a currency string (`$X.XX`).
- Display all 9 columns without truncation, using a scoped context manager without mutating the underlying numbers!

In [6]:
# Interview Solution: Styler Formatting (Zero Mutation of Underlying Data)
styled_dashboard = (
    happiness.head(5).style
    .format({
        "Score": "{:.2f} ★",
        "GDP per capita": "${:,.2f}",
        "Social support": "{:.1%}",
        "Life expectancy": "{:.2f} yrs"
    })
    .set_caption("Executive World Happiness Index (Custom Styled Presentation)")
)

print("Rendered Styler Dashboard (Underlying data types remain untouched):")
display(styled_dashboard)

Rendered Styler Dashboard (Underlying data types remain untouched):


,Country,Score,GDP per capita,Social support,Life expectancy,Generosity
0,Finland,7.77 ★,$1.34,158.7%,0.99 yrs,0.153000
1,Denmark,7.60 ★,$1.38,157.3%,1.00 yrs,0.252000
2,Norway,7.55 ★,$1.49,158.2%,1.03 yrs,0.271000
3,Iceland,7.49 ★,$1.38,162.4%,1.03 yrs,0.354000
4,Netherlands,7.49 ★,$1.40,152.2%,1.00 yrs,0.322000
